# Used Car Price Prediction

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings

warnings.filterwarnings("ignore")

%matplotlib inline

In [2]:
df = pd.read_csv('/home/ashish/Desktop/All/Algorithms/11_XgBoost/cardekho_imputated.csv')

In [3]:
df.isnull().sum()

Unnamed: 0           0
car_name             0
brand                0
model                0
vehicle_age          0
km_driven            0
seller_type          0
fuel_type            0
transmission_type    0
mileage              0
engine               0
max_power            0
seats                0
selling_price        0
dtype: int64

In [4]:
print(df.columns.tolist())

['Unnamed: 0', 'car_name', 'brand', 'model', 'vehicle_age', 'km_driven', 'seller_type', 'fuel_type', 'transmission_type', 'mileage', 'engine', 'max_power', 'seats', 'selling_price']


In [5]:
df.drop('car_name',axis=1,inplace= True)
df.drop('brand',axis=1,inplace = True)
df.drop(columns='Unnamed: 0', errors='ignore', inplace=True)

In [6]:
df['model'].unique()

<StringArray>
[        'Alto',        'Grand',          'i20',     'Ecosport',
      'Wagon R',          'i10',        'Venue',        'Swift',
        'Verna',       'Duster',
 ...
     'Panamera',      'Alturas',       'Altroz',           'NX',
     'Carnival',            'C',           'RX',        'Ghost',
 'Quattroporte',       'Gurkha']
Length: 120, dtype: str

In [7]:
## Getting All Different Types OF Features
num_features = [features for features in df.columns if df[features].dtype !='int']
print("Num of Numerical features :",len(num_features))

cat_features =[features for features in df.columns if df[features].dtype =='str']
print('num of categrical Features  :',len(cat_features))

Num of Numerical features : 6
num of categrical Features  : 4


In [8]:
df.head()

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [9]:
## Now Selecting the depended and independed Features 
from sklearn.model_selection import train_test_split
X=df.drop(['selling_price'],axis=1)
y=df['selling_price']

In [10]:
len(df['model'].unique())

120

In [11]:
df['model'].value_counts()

model
i20             906
Swift Dzire     890
Swift           781
Alto            778
City            757
               ... 
Altroz            1
C                 1
Ghost             1
Quattroporte      1
Gurkha            1
Name: count, Length: 120, dtype: int64

In [12]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

In [13]:
X['model']=le.fit_transform(X['model'])

In [14]:
X['model']

0          7
1         54
2        118
3          7
4         38
        ... 
15406    117
15407     42
15408     77
15409    114
15410     25
Name: model, Length: 15411, dtype: int64

In [15]:
## Devidind the data according to the categries
cat_features = ['seller_type','fuel_type','transmission_type']
num_features = X.select_dtypes(exclude='object').columns

from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer

Scaler_transform = StandardScaler()
OneHo_transform = OneHotEncoder(drop='first')

proccess = ColumnTransformer(
    [
        ('onrhot',OneHo_transform,cat_features),
        ('Scaler',Scaler_transform,num_features)
    ],remainder='passthrough'
)

In [16]:
print(df.columns.tolist())

['model', 'vehicle_age', 'km_driven', 'seller_type', 'fuel_type', 'transmission_type', 'mileage', 'engine', 'max_power', 'seats', 'selling_price']


In [17]:
X = proccess.fit_transform(X)

In [18]:
## Now i will spliting the train and test
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)
X_train.shape, X_test.shape

((12328, 14), (3083, 14))

In [19]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.linear_model import LinearRegression,Ridge, Lasso
from  sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRFRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [20]:
## Creat the Function for the evaluting the
def evalute_model(true,predict):
    mae= mean_absolute_error(true,predict)
    mse= mean_squared_error(true,predict)
    rmse=np.sqrt(mean_squared_error(true,predict))
    r2_square= r2_score(true,predict)
    return mae,rmse,r2_square

In [21]:
## Bigigng the Model training
models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "K-Neighbors Regressor": KNeighborsRegressor(),
    "XG -Boost":XGBRFRegressor(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest Regressor": RandomForestRegressor(),
    "Adaboost":AdaBoostRegressor()
   
}

for i in range (len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train,y_train)

    ## Make Prediction
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    ## Evalution of dataset
    model_train_mae, model_train_rmse, model_train_r2 = evalute_model(y_train,y_pred_train)

    model_test_mae, model_test_rmse, model_test_r2 = evalute_model(y_test,y_pred_test)

    print(list(models.keys())[i])
    
    print('Model performance for Training set')
    print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_train_mae))
    print("- R2 Score: {:.4f}".format(model_train_r2))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
    print("- R2 Score: {:.4f}".format(model_test_r2))
    
    print('='*35)
    print('\n')

Linear Regression
Model performance for Training set
- Root Mean Squared Error: 553855.6665
- Mean Absolute Error: 268101.6071
- R2 Score: 0.6218
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 502543.5930
- Mean Absolute Error: 279618.5794
- R2 Score: 0.6645


Lasso
Model performance for Training set
- Root Mean Squared Error: 553855.6710
- Mean Absolute Error: 268099.2226
- R2 Score: 0.6218
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 502542.6696
- Mean Absolute Error: 279614.7461
- R2 Score: 0.6645


Ridge
Model performance for Training set
- Root Mean Squared Error: 553856.3160
- Mean Absolute Error: 268059.8015
- R2 Score: 0.6218
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 502533.8230
- Mean Absolute Error: 279557.2169
- R2 Score: 0.6645


K-Neighbors Regressor
Model performance for Training set
- Root Mean Squared Error: 325886.0010
- Mean 

In [22]:
## Hyperparameter Tuning 
rf_parms = {
    'max_depth':[3,4,5,8,10,12,None],
    'max_features':[3,5,8,7],
    'min_samples_split':[2,8,15,20,4],
    'n_estimators':[100,200,500,1000]
}

XgBoost_params = {
    'n_estimators'    : [100, 200, 500],
    'learning_rate'   : [0.01, 0.05, 0.1],
    'max_depth'       : [3, 5,8,12],
    'colsample_bytree': [0.7, 1.0,0.3,0.4]
}


In [23]:
## Model list for Hyper perameter Tunning
randomcv_parms = [
    ("RF", RandomForestRegressor(), rf_parms),
    ('XG',XGBRFRegressor(),XgBoost_params)
]

In [24]:
from sklearn.model_selection import RandomizedSearchCV
models_parms={}

for name,model,parms in randomcv_parms:
    random =RandomizedSearchCV(estimator=model,
                                param_distributions=parms,
                                n_iter=100,
                                cv=3,
                                verbose=2,
                                n_jobs=-1)
    random.fit(X_train,y_train)
    models_parms[name]=random.best_params_

    for model_name in models_parms:
        print(f"---------------- Best Params for {model_name} -------------------")
        print(models_parms[model_name])

    

Fitting 3 folds for each of 100 candidates, totalling 300 fits
[CV] END max_depth=None, max_features=3, min_samples_split=20, n_estimators=200; total time=   1.2s
[CV] END max_depth=4, max_features=3, min_samples_split=20, n_estimators=200; total time=   0.6s
[CV] END max_depth=4, max_features=3, min_samples_split=20, n_estimators=200; total time=   0.7s
[CV] END max_depth=4, max_features=3, min_samples_split=20, n_estimators=200; total time=   1.0s
[CV] END max_depth=None, max_features=3, min_samples_split=20, n_estimators=200; total time=   1.4s
[CV] END max_depth=None, max_features=3, min_samples_split=20, n_estimators=200; total time=   1.8s
[CV] END max_depth=4, max_features=8, min_samples_split=15, n_estimators=200; total time=   1.0s
[CV] END max_depth=12, max_features=7, min_samples_split=15, n_estimators=200; total time=   2.2s
[CV] END max_depth=4, max_features=8, min_samples_split=15, n_estimators=200; total time=   1.2s
[CV] END max_depth=4, max_features=8, min_samples_spli

In [31]:
## Retrain the model with parms
models = {
    "Random Forest Reegression": RandomForestRegressor(n_estimators=100,# Zyada trees
                                                       min_samples_split=2,
                                                       max_features=7,
                                                       max_depth=None,
                                                       n_jobs=-1),
    "Gradient-Boosting" :  XGBRFRegressor(n_estimators=500,
                                                      max_depth= 12, learning_rate= 0.1,colsample_bytree=1.0 )                                                   
                                                                                                                                                               
}

for i in range(len(list(models))):
    model=list(models.values())[i]
    model.fit(X_train,y_train)

     # Make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    model_train_mae , model_train_rmse, model_train_r2 = evalute_model(y_train, y_train_pred)

    model_test_mae , model_test_rmse, model_test_r2 = evalute_model(y_test, y_test_pred)
    
    print(list(models.keys())[i])
    
    print('Model performance for Training set')
    print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_train_mae))
    print("- R2 Score: {:.4f}".format(model_train_r2))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
    print("- R2 Score: {:.4f}".format(model_test_r2))
    
    print('='*35)
    print('\n')


Random Forest Reegression
Model performance for Training set
- Root Mean Squared Error: 143880.3812
- Mean Absolute Error: 39605.0916
- R2 Score: 0.9745
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 219021.2242
- Mean Absolute Error: 98830.8765
- R2 Score: 0.9363


Gradient-Boosting
Model performance for Training set
- Root Mean Squared Error: 814734.2277
- Mean Absolute Error: 403817.4062
- R2 Score: 0.1815
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 779101.2918
- Mean Absolute Error: 423119.6875
- R2 Score: 0.1937


